### Librerias

In [46]:
import pandas as pd
from pathlib import Path
from utils import guardar_pickle, rango_edad, perfil_marca


### Carga datos: CSV

In [47]:
BASE_DIR = Path.cwd()          # Proyecto/src
DATA_PATH = BASE_DIR / "data" / "raw" / "data-CCS.csv"

df = pd.read_csv(DATA_PATH)
df['id'] = range(1, len(df) + 1)

### Limpieza

In [48]:
# Asignación de nuevo nombre a las columnas necesarias
df.rename(columns={'Ciudad de residencia':'ciudad','Nivel de estudios':'estudios',
                    '¿Dónde sueles comprar tus productos cosméticos?':'lugar_compra',
                    'Gasto promedio en €':'gasto_avg',
                    '¿Cuánto vale el producto más caro que has comprado en el último año?':'producto_caro',
                    '¿Cuántos productos utilizas en tu día a día?':'cant_productos_uso',
                    'Frecuencia de compra cosmética':'frecuencia',
                    '¿Lees el INCI/etiqueta del producto?':'INCI',
                    '¿Usas algún medio para profundizar más en tus conocimientos cosméticos?':'medio_si_no',
                    '¿Qué medio es el que más usas?':'medio',
                    '¿En qué medida te consideras bien informade sobre ingredientes de los productos cosméticos? (1-Nada, 5-Mucho)':'calificacion_informado',
                    '¿Consideras que etiquetas como "clean/eco/natural" garantizan un producto de mejor calidad?':'eco_natural',
                    '¿Conoces las certificaciones? (Ej: Ecocert, Cruelty-free, etc)':'certificaciones_si_no',
                    '¿Qué certificaciones conoces? Ej: Vegan, Cruelty-free, etc':'certificaciones_conocidas',
                    '¿Por qué marca te decantarías antes?':'Estilo_marca',
                    'Valora del 1 (nada) al 5 (muy importante) el peso de los siguientes criterios cuando compras: [Precio]':'precio',
                    'Valora del 1 (nada) al 5 (muy importante) el peso de los siguientes criterios cuando compras: [Marca]':'marca',
                    'Valora del 1 (nada) al 5 (muy importante) el peso de los siguientes criterios cuando compras: [Ingredientes]':'ingredientes',
                    'Valora del 1 (nada) al 5 (muy importante) el peso de los siguientes criterios cuando compras: [Packaging/diseño envase]':'packaging',
                    'Valora del 1 (nada) al 5 (muy importante) el peso de los siguientes criterios cuando compras: [Certificaciones]':'certificaciones',
                    'Valora del 1 (nada) al 5 (muy importante) el peso de los siguientes criterios cuando compras: [Reviews/opiniones]':'reviews',
                    'Valora del 1 (nada) al 5 (muy importante) el peso de los siguientes criterios cuando compras: [Influencer/viral en redes]':'viral',
                    '¿Has comprado alguna vez un producto por recomendación de influencer?':'influencer',
                    '¿Qué criterios te harían dejar de comprar una marca? (puedes marcar varias)':'criterios'
                    }, inplace=True)

df.drop(columns=['Marca temporal'], inplace=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166 entries, 0 to 165
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Edad                       166 non-null    int64 
 1   Sexo                       166 non-null    object
 2   ciudad                     166 non-null    object
 3   estudios                   166 non-null    object
 4   frecuencia                 166 non-null    object
 5   gasto_avg                  166 non-null    object
 6   producto_caro              166 non-null    object
 7   cant_productos_uso         166 non-null    int64 
 8   lugar_compra               166 non-null    object
 9   Estilo_marca               166 non-null    object
 10  INCI                       166 non-null    object
 11  certificaciones_si_no      45 non-null     object
 12  precio                     166 non-null    int64 
 13  marca                      166 non-null    int64 
 14  ingredient

#### Creación de columnas

In [49]:
df['rango_edad'] = df['Edad'].apply(rango_edad)
df['preferencia'] = df['Estilo_marca'].apply(perfil_marca)

perfil_dummies = pd.get_dummies(df['preferencia'], prefix='preferencia')
df = pd.concat([df, perfil_dummies], axis=1)


df_lugar = pd.get_dummies(df['lugar_compra'], prefix='lugar')
df = pd.concat([df, df_lugar], axis=1)

df['bcn_s_n'] = df['ciudad'].apply(    lambda c: 'BCN' if isinstance(c, str) and c.strip().lower() == 'barcelona' else 'Otros')

#### Mapeo de variables

In [50]:
gasto_avg_map = {
    "0-20€": 10,
    "21-40€": 30,
    "41-60€": 50,
    "61-100€": 80,
    "Más de 100€": 120  
}
df["gasto_por_compra"] = df["gasto_avg"].map(gasto_avg_map)

compras_mes = {
    "Semanalmente": 4,
    "Mensualmente": 1,
    "Trimestralmente": 1/3,       
    "Ocasionalmente/anualmente": 2/12  
}
df["compras_por_mes"] = round(df["frecuencia"].map(compras_mes),2)

# Calcular gasto mensual promedio
df["gasto_mensual_avg"] = round(df["gasto_por_compra"] * df["compras_por_mes"],2)

frec_map = {
    "Ocasionalmente/anualmente": 1,
    "Trimestralmente": 2,
    "Mensualmente": 3,
    "Semanalmente": 4
}
df["freq_score"] = round(df["frecuencia"].map(frec_map),2)

inci_map = {
    "Siempre": 2,
    "A veces": 1,
    "Nunca": 0
}
df["inci_score"] = df["INCI"].map(inci_map)

medios_map = {
    "Sí":1,
    "No":0
}
df["medios_score"] = df["medio_si_no"].map(medios_map)

eco_map = {
    "Si": 0,
    "No": 0,
    "Tal vez": 1
}
df["eco_natural"] = df["eco_natural"].fillna('No')
df["eco_score"] = df["eco_natural"].map(eco_map)

binarias = [
    "preferencia_Dermo",
    "preferencia_Lujo",
    "preferencia_Natural",
    "preferencia_Trendy",
    "lugar_Farmacia",
    "lugar_Online"
]

for col in binarias:
    df[col] = df[col].map({True:1, False:0})

df['influencer'] = df['influencer'].fillna('No')

df['influ_score'] = df['influencer'].map({'Sí': 1, 'No': 0}).astype(int)

df["producto_caro"] = (
    df["producto_caro"]
    .str.replace("€", "", regex=False)
    .str.replace("euros", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
    .round(2)
)

completo = df
completo

,Edad,Sexo,ciudad,estudios,frecuencia,gasto_avg,producto_caro,cant_productos_uso,lugar_compra,Estilo_marca,...,lugar_Tienda,bcn_s_n,gasto_por_compra,compras_por_mes,gasto_mensual_avg,freq_score,inci_score,medios_score,eco_score,influ_score
0,37,Mujer,Barcelona,Universidad/Grado,Ocasionalmente/anualmente,41-60€,58.0,3,Online,Avene,...,False,BCN,50,0.17,8.5,1,1,1,0.0,0
1,36,Mujer,Barcelona,Formación profesional/técnica,Trimestralmente,21-40€,35.0,4,Tienda,Shiseido,...,True,BCN,30,0.33,9.9,2,2,1,0.0,0
2,30,Mujer,Vilanova i la Geltru,Postgrado,Trimestralmente,41-60€,54.0,4,Tienda,Cerave,...,True,Otros,50,0.33,16.5,2,2,0,0.0,0
3,40,Hombre,Barcelona,Universidad/Grado,Ocasionalmente/anualmente,0-20€,5.0,1,Farmacia,Dior,...,False,BCN,10,0.17,1.7,1,1,0,1.0,0
4,27,Mujer,Barcelona,Universidad/Grado,Mensualmente,41-60€,65.0,4,Tienda,Avene,...,True,BCN,50,1.00,50.0,3,2,0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,26,Mujer,Santiago de Compostela,Postgrado,Trimestralmente,41-60€,30.0,3,Tienda,Avene,...,True,Otros,50,0.33,16.5,2,1,1,0.0,0
162,25,Mujer,Barcelona,Postgrado,Trimestralmente,21-40€,25.0,7,Tienda,Medicube,...,True,BCN,30,0.33,9.9,2,2,1,0.0,0
163,33,Mujer,Sevilla,Postgrado,Trimestralmente,61-100€,70.0,5,Farmacia,Lush,...,False,Otros,80,0.33,26.4,2,1,0,0.0,0
164,21,Mujer,Madrid,Secundaria/Bachillerato,Mensualmente,0-20€,20.0,2,Online,Cerave,...,False,Otros,10,1.00,10.0,3,0,1,0.0,0


### Creación de dataframes

##### Datos demográficos

In [51]:
personas = completo[['id','Edad','rango_edad','Sexo','ciudad','bcn_s_n','estudios','lugar_compra','frecuencia']]
personas

,id,Edad,rango_edad,Sexo,ciudad,bcn_s_n,estudios,lugar_compra,frecuencia
0,1,37,34-38,Mujer,Barcelona,BCN,Universidad/Grado,Online,Ocasionalmente/anualmente
1,2,36,34-38,Mujer,Barcelona,BCN,Formación profesional/técnica,Tienda,Trimestralmente
2,3,30,29-33,Mujer,Vilanova i la Geltru,Otros,Postgrado,Tienda,Trimestralmente
3,4,40,39-43,Hombre,Barcelona,BCN,Universidad/Grado,Farmacia,Ocasionalmente/anualmente
4,5,27,24-28,Mujer,Barcelona,BCN,Universidad/Grado,Tienda,Mensualmente
...,...,...,...,...,...,...,...,...,...
161,162,26,24-28,Mujer,Santiago de Compostela,Otros,Postgrado,Tienda,Trimestralmente
162,163,25,24-28,Mujer,Barcelona,BCN,Postgrado,Tienda,Trimestralmente
163,164,33,29-33,Mujer,Sevilla,Otros,Postgrado,Farmacia,Trimestralmente
164,165,21,<24,Mujer,Madrid,Otros,Secundaria/Bachillerato,Online,Mensualmente


#### Consumo

In [52]:
consumo = completo[['id','gasto_avg','producto_caro','gasto_mensual_avg','gasto_por_compra','cant_productos_uso']]
consumo

,id,gasto_avg,producto_caro,gasto_mensual_avg,gasto_por_compra,cant_productos_uso
0,1,41-60€,58.0,8.5,50,3
1,2,21-40€,35.0,9.9,30,4
2,3,41-60€,54.0,16.5,50,4
3,4,0-20€,5.0,1.7,10,1
4,5,41-60€,65.0,50.0,50,4
...,...,...,...,...,...,...
161,162,41-60€,30.0,16.5,50,3
162,163,21-40€,25.0,9.9,30,7
163,164,61-100€,70.0,26.4,80,5
164,165,0-20€,20.0,10.0,10,2


#### Información

In [53]:
informado = completo[['id','INCI','medio_si_no','medio','calificacion_informado','eco_natural','certificaciones_conocidas']]
informado


,id,INCI,medio_si_no,medio,calificacion_informado,eco_natural,certificaciones_conocidas
0,1,A veces,Sí,"Internet (blogs, videos, etc)",4,No,NaN
1,2,Siempre,Sí,"Internet (blogs, videos, etc)",4,No,NaN
2,3,Siempre,No,NaN,2,No,NaN
3,4,A veces,No,NaN,1,Tal vez,NaN
4,5,Siempre,No,NaN,4,No,NaN
...,...,...,...,...,...,...,...
161,162,A veces,Sí,"Internet (blogs, videos, etc)",3,No,"Cruelty-free, Vegan"
162,163,Siempre,Sí,"Internet (blogs, videos, etc)",5,No,"Cruelty-free, COSMOS, Vegan, ISO"
163,164,A veces,No,NaN,2,No,"Cruelty-free, Vegan"
164,165,Nunca,Sí,"Internet (blogs, videos, etc)",3,No,"Cruelty-free, Vegan"


#### Influencia

In [54]:
influencia = completo[['id','Estilo_marca','preferencia','precio','ingredientes','viral','packaging','marca','reviews','certificaciones','influencer','criterios']]
influencia.fillna({'influencer':'No'}, inplace=True)
influencia

C:\Users\Usuario\AppData\Local\Temp\ipykernel_40412\888068575.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  influencia.fillna({'influencer':'No'}, inplace=True)


,id,Estilo_marca,preferencia,precio,ingredientes,viral,packaging,marca,reviews,certificaciones,influencer,criterios
0,1,Avene,Dermo,5,4,2,3,4,4,3,No,"Subida de precio, Mala reputación de la marca/..."
1,2,Shiseido,Lujo,3,5,4,3,4,4,2,No,"Cambio de formulación, Subida de precio, Reseñ..."
2,3,Cerave,Dermo,4,4,3,4,2,5,2,No,"Cambio de formulación, Subida de precio, Uso d..."
3,4,Dior,Lujo,3,1,1,1,1,1,1,No,"Subida de precio, Mala reputación de la marca/..."
4,5,Avene,Dermo,3,3,2,1,3,3,1,No,"Cambio de formulación, Subida de precio, Mala ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
161,162,Avene,Dermo,3,4,1,2,4,4,2,No,"Cambio de formulación, Subida de precio, Mala ..."
162,163,Medicube,Trendy,5,5,2,4,4,4,1,No,"Cambio de formulación, Subida de precio, Reseñ..."
163,164,Lush,Natural,4,5,1,3,3,3,3,No,"Cambio de formulación, Subida de precio, Recom..."
164,165,Cerave,Dermo,5,5,2,3,4,4,3,No,"Subida de precio, Mala reputación de la marca/..."


### Creación de dimensiones/tablas puente

#### Certificaciones

In [55]:
# Creación de tabla puente entre las respuestas y la dimensión de certificaciones: tener cada id con su selección múltiple por separado
tab_cert = informado[["id", "certificaciones_conocidas"]].dropna().copy()
tab_cert["certificacion"] = tab_cert["certificaciones_conocidas"].str.split(",")
tab_cert = tab_cert.explode("certificacion")

# Creación de la dimensión de las certificaciones: tener cada certificación con un id para posteriores relaciones
dim_cert = (
    tab_cert[["certificacion"]]
    .drop_duplicates()
    .sort_values("certificacion")
    .reset_index(drop=True)
)

dim_cert["id_certificacion"] = dim_cert.index + 1
dim_cert = dim_cert[["id_certificacion", "certificacion"]]

# Actualización de la tabla puente para que únicamente hayan los id que relacionan la tabla de respuestas con la dimensión de certificaciones
tab_cert = tab_cert[["id", "certificacion"]].rename(
    columns={"id": "id_respuesta"}
)
tab_cert

,id_respuesta,certificacion
100,101,Vegan
100,101,ISO
101,102,Vegan
101,102,ISO
102,103,EcoCert
...,...,...
164,165,Vegan
165,166,EcoCert
165,166,Cruelty-free
165,166,Vegan


#### Criterios

In [56]:
# Creación dimensión de criterior para tenerlas por separado según id de respuesta
tab_criterios = influencia[["id", "criterios"]].dropna().copy()
tab_criterios["criterio"] = tab_criterios["criterios"].str.split(",")
tab_criterios = tab_criterios.explode("criterio")
tab_criterios["criterio"] = tab_criterios["criterio"].str.strip()

# Creación de la dimensión de los criterios: tener cada criterio con un id para posteriores relaciones
dim_criterios = (
    tab_criterios[["criterio"]]
    .drop_duplicates()
    .sort_values("criterio")
    .reset_index(drop=True)
)

dim_criterios["id_criterio"] = dim_criterios.index + 1
dim_criterios = dim_criterios[["id_criterio", "criterio"]]

# Actualización de la tabla puente para que únicamente hayan los id que relacionan la tabla de respuestas con la dimensión de certificaciones
tab_criterios = tab_criterios[["id", "criterio"]].rename(
    columns={"id": "id_respuesta"}
)
tab_criterios

,id_respuesta,criterio
0,1,Subida de precio
0,1,Mala reputación de la marca/responsabilidad so...
0,1,Reseñas negativas
0,1,Uso de marketing engañoso
1,2,Cambio de formulación
...,...,...
165,166,Cambio de formulación
165,166,Subida de precio
165,166,Mala reputación de la marca/responsabilidad so...
165,166,Uso de marketing engañoso


### Guardar dataframes completos

In [57]:
clusters = pd.read_pickle('./data/clean/clusters.pkl')
pca = pd.read_pickle('./data/clean/pca.pkl')
output_dir = Path("data/clean")
output_dir.mkdir(parents=True, exist_ok=True)

dataframes = {
    "completo": completo,
    "usuarios": personas,
    "consumo": consumo,
    "influencia": influencia,
    "informado": informado,
    "tab_cert": tab_cert,
    "tab_criterios": tab_criterios,
    "clusters" : clusters,
    "pca": pca
}

for nombre, df in dataframes.items():
    if df is None:
        raise ValueError(f"{nombre} es None")
    if df.empty:
        print(f"Advertencia: {nombre} está vacío")
    guardar_pickle(df,f"{nombre}.pkl")
    df.to_csv(output_dir/f"{nombre}.csv", index=False, encoding="utf-8-sig",decimal=",")


DataFrame guardado en: C:\Users\Usuario\Desktop\DataAnalytics\Proyecto\src\data\clean\completo.pkl
DataFrame guardado en: C:\Users\Usuario\Desktop\DataAnalytics\Proyecto\src\data\clean\usuarios.pkl
DataFrame guardado en: C:\Users\Usuario\Desktop\DataAnalytics\Proyecto\src\data\clean\consumo.pkl
DataFrame guardado en: C:\Users\Usuario\Desktop\DataAnalytics\Proyecto\src\data\clean\influencia.pkl
DataFrame guardado en: C:\Users\Usuario\Desktop\DataAnalytics\Proyecto\src\data\clean\informado.pkl
DataFrame guardado en: C:\Users\Usuario\Desktop\DataAnalytics\Proyecto\src\data\clean\tab_cert.pkl
DataFrame guardado en: C:\Users\Usuario\Desktop\DataAnalytics\Proyecto\src\data\clean\tab_criterios.pkl
DataFrame guardado en: C:\Users\Usuario\Desktop\DataAnalytics\Proyecto\src\data\clean\clusters.pkl
DataFrame guardado en: C:\Users\Usuario\Desktop\DataAnalytics\Proyecto\src\data\clean\pca.pkl
